In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # Disable HF transfer to avoid timeout issues
os.environ["HUGGINGFACE_HUB_VERBOSITY"] = "debug"  # Enable debug logging for HuggingFace hub

import gc  # Garbage collector for memory management
import glob  # File pattern matching
import torch  # PyTorch deep learning framework
import unsloth  # Unsloth optimization library
from unsloth import FastLanguageModel  # Fast language model wrapper
from trl import SFTTrainer, SFTConfig  # Supervised Fine-Tuning trainer and config
from transformers import TrainingArguments  # Training arguments from HuggingFace transformers
from datasets import load_dataset  # Dataset loading utilities

torch.cuda.empty_cache()  # Clear CUDA cache to free GPU memory
print(f"PyTorch version: {torch.__version__}")  # Display installed PyTorch version
print(f"CUDA available:  {torch.cuda.is_available()}")  # Check if CUDA (GPU support) is available
print(f"GPU:             {torch.cuda.get_device_name(0)}")  # Print GPU device name
print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")  # Display total GPU memory in GB

BASE_DIR = os.getcwd()  # Set BASE_DIR to current working directory
print(f"BASE_DIR: {BASE_DIR}")  # Print the base directory path for verification

os.makedirs(os.path.join(BASE_DIR, "ollama_phi_chat"), exist_ok=True)  # Create output directory (skip if already exists)

In [10]:
# ── 1. Load base model ────────────────────────────────────────────────────────
print("Loading Phi-3-mini-4k-instruct...")  # Status message
model, tokenizer = FastLanguageModel.from_pretrained(  # Load pre-trained Phi-3 model from HuggingFace
    model_name="microsoft/Phi-3-mini-4k-instruct",  # Microsoft's Phi-3 mini model (4K context)
    device_map="cuda",  # Load model on GPU
    torch_dtype="auto",  # Automatically determine optimal data type
    trust_remote_code=True,  # Allow custom code from HuggingFace repo
    load_in_4bit=True  # Quantize model to 4-bit for memory efficiency (fits in 4GB VRAM)
)

In [11]:
# ── 2. Apply LoRA ─────────────────────────────────────────────────────────────
print("Applying LoRA...")  # Status message
model = FastLanguageModel.get_peft_model(  # Wrap model with LoRA (Low-Rank Adaptation) for efficient fine-tuning
    model,  # Base model to apply LoRA to
    r=8,  # Rank of LoRA matrices (lower = more memory efficient, kept low for 4GB VRAM)
    lora_alpha=8,  # LoRA scaling factor (typically matches rank to avoid scaling issues)
    lora_dropout=0.05,  # Dropout rate for LoRA layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",  # Apply LoRA to attention projection layers
                    "gate_proj", "up_proj", "down_proj"],  # Apply LoRA to MLP layers
    use_gradient_checkpointing="unsloth",  # Enable gradient checkpointing to save memory
    random_state=42,  # Random seed for reproducibility
)

Applying LoRA...


In [12]:

# ── 3. Load and format dataset ────────────────────────────────────────────────
print("Loading dataset...")  # Status message
dataset = load_dataset("json",  # Load dataset from JSON file format
    data_files=os.path.join(BASE_DIR, "dotnet_azure_ai_dataset.json"))  # Path to training data JSON file
train_data = dataset["train"]  # Extract train split from loaded dataset

# FIX: use tokenizer's chat template — handles <|end|> tokens correctly
def format_instruction(example):  # Convert raw data to chat-formatted text
    return {  # Return dictionary with formatted text
        "text": tokenizer.apply_chat_template(  # Apply tokenizer's chat template
            example["messages"],  # Input messages list
            tokenize=False,  # Don't tokenize, just format as string
            add_generation_prompt=False,  # Don't add generation prompt tokens
        )
    }

train_dataset = train_data.map(format_instruction)  # Apply formatting function to all samples

FastLanguageModel.for_training(model)  # Prepare model for training mode (enables gradients for LoRA params)

Loading dataset...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
   

In [13]:
# ── 4. Train ──────────────────────────────────────────────────────────────────
torch.cuda.empty_cache()  # Clear GPU memory before training
print("Starting training...")  # Status message

trainer = SFTTrainer(  # Create Supervised Fine-Tuning trainer
    model=model,  # Model with LoRA to fine-tune
    tokenizer=tokenizer,  # Tokenizer for processing text
    train_dataset=train_dataset,  # Pre-formatted training data
    args=SFTConfig(  # Training configuration
        output_dir="phichat",  # Directory to save checkpoints
        num_train_epochs=3,  # Train for 3 epochs
        per_device_train_batch_size=1,  # Batch size 1 (limited by 4GB VRAM)
        gradient_accumulation_steps=8,  # Accumulate gradients over 8 steps to simulate batch size of 8
        warmup_steps=10,  # Warm up learning rate over 10 steps
        learning_rate=2e-4,  # Learning rate for LoRA parameters
        bf16=True,  # Use bfloat16 precision (faster training, less memory)
        fp16=False,  # Don't use float16 (conflicts with bf16)
        logging_steps=10,  # Log metrics every 10 steps
        eval_strategy="no",  # Don't evaluate during training (save memory)
        save_strategy="epoch",  # Save checkpoint after each epoch
        save_total_limit=1,  # Keep only 1 checkpoint (save disk space)
        optim="adamw_8bit",  # Use 8-bit AdamW optimizer (memory efficient)
        max_grad_norm=0.3,  # Clip gradients to prevent instability
        dataloader_pin_memory=False,  # Don't pin memory (can cause CUDA issues)
        dataloader_num_workers=0,  # Single process data loading
        remove_unused_columns=False,  # Keep all columns in dataset
        report_to="none",  # Don't report to external services (no wandb/tensorboard)
        gradient_checkpointing=True,  # Enable gradient checkpointing to save memory
        # FIX: these two must live in SFTConfig, not as SFTTrainer kwargs
        dataset_text_field="text",  # Column name containing training text
        max_seq_length=512,  # Max sequence length (shorter = faster, less memory)
    ),
)

try:  # Error handling for training
    trainer.train()  # Start fine-tuning
    print("Training completed successfully.")  # Success message
except RuntimeError as e:  # Catch out-of-memory or CUDA errors
    if "memory" in str(e).lower() or "cuda" in str(e).lower():  # Check if OOM error
        print(f"OOM: {e}")  # Print OOM error
        print("Open the script and reduce max_seq_length to 256 then retry.")  # Recovery suggestion
    else:  # Handle other runtime errors
        import traceback  # Import traceback for detailed error info
        traceback.print_exc()  # Print full error trace

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 81 | Num Epochs = 3 | Total steps = 33
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 14,942,208 of 3,836,021,760 (0.39% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.663100
20,1.154200
30,0.566700


Training completed successfully.


In [14]:
# ── 5. Save LoRA adapters ─────────────────────────────────────────────────────
print("Saving LoRA adapters...")  # Status message
model.save_pretrained(os.path.join(BASE_DIR, "ollama_phi_chat"))  # Save LoRA adapter weights to directory
tokenizer.save_pretrained(os.path.join(BASE_DIR, "ollama_phi_chat"))  # Save tokenizer files (vocab, config)
print("Saved to " + os.path.join(BASE_DIR, "ollama_phi_chat"))  # Confirm save location

Saving LoRA adapters...
Saved to d:\AI_Projects\Unsloth_Phi3_Ollama\dotnet-ai-assistant\dotnet-assistant-trainer\ollama_phi_chat


In [ ]:
# ── 6. Export GGUF ────────────────────────────────────────────────────────────

print("Exporting GGUF...")  # Status message
del trainer  # Delete trainer object to free memory
torch.cuda.empty_cache()  # Clear GPU cache
gc.collect()  # Run garbage collection to free RAM
torch.cuda.empty_cache()  # Clear GPU cache again after garbage collection

FastLanguageModel.for_inference(model)  # Switch model to inference mode (disable gradient computation)

EXPORT_DIR = "ollama_phi_chat"  # Base export directory name (Unsloth will create ollama_phi_chat_gguf)

try:  # Error handling for GGUF export
    model.save_pretrained_gguf(  # Export model to GGUF format (optimized for Ollama)
        EXPORT_DIR,  # Base directory (NOT "ollama_phi_chat_gguf")
        tokenizer,  # Tokenizer to include in export
        quantization_method="q8_0",  # Use q8_0 quantization (aggressive, ~8 bits per weight)
    )
    print("GGUF export complete.")  # Success message
except RuntimeError as e:  # Catch export failures
    print(f"GGUF export failed: {e}")  # Print error message
    print("Restart kernel, reload model+adapters, then run only the export cell.")  # Recovery steps

Exporting GGUF...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: C:\Users\soumy\.cache\huggingface\hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 165.36it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:24<00:00, 12.19s/it]


Unsloth: Merge process complete. Saved to `d:\AI_Projects\Unsloth_Phi3_Ollama\dotnet-ai-assistant\dotnet-assistant-trainer\ollama_phi_chat`


Unsloth: Extending ollama_phi_chat/tokenizer.model with added_tokens.json.
Originally tokenizer.model is of size (32000).
But we need to extend to sentencepiece vocab size (32011).


Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
GGUF export failed: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model to GGUF with command `d:\AI_Projects\Unsloth_Phi3_Ollama\dotnet-ai-assistant\dotnet-assistant-trainer\.venv\Scripts\python.exe C:\Users\soumy\.unsloth\llama.cpp\unsloth_convert_hf_to_gguf.py --outfile phi-3-mini-4k-instruct.BF16.gguf --outtype bf16 --split-max-size 50G ollama_phi_chat`: Command '['d:\\AI_Projects\\Unsloth_Phi3_Ollama\\dotnet-ai-assistan

In [16]:
# ── 7. Find GGUF and write Modelfile ─────────────────────────────────────────
# Unsloth creates: {EXPORT_DIR}_gguf/
expected_dir = f"{EXPORT_DIR}_gguf"  # Expected directory is "ollama_phi_chat_gguf"
gguf_files = glob.glob(f"{expected_dir}/*.gguf")  # Find all .gguf files in expected directory


# Fallback: scan all subfolders in case Unsloth version behaves differently
if not gguf_files:  # If not found in expected location
    print(f"\nNot found in expected folder '{expected_dir}/', scanning all folders...")  # Status message
    for d in os.listdir("."):  # Loop through all items in current directory
        if os.path.isdir(d):  # Check if item is a directory
            found = glob.glob(f"{d}/*.gguf")  # Search for .gguf files in this directory
            if found:  # If .gguf files found
                gguf_files = found  # Store the found files
                print(f"Found GGUF in: {d}/")  # Print location
                break  # Stop searching

if not gguf_files:  # If no GGUF files found anywhere
    print("\nERROR: No GGUF file found anywhere.")  # Error message
    print("Folders present:")  # Print debug info header
    for d in sorted(os.listdir(".")):  # List all directories
        if os.path.isdir(d):  # Filter to directories only
            contents = os.listdir(d)  # Get directory contents
            print(f"  {d}/  ({len(contents)} files)")  # Print directory and file count
else:  # If GGUF files found
    gguf_path = os.path.abspath(gguf_files[0]).replace("\\", "/")  # Get absolute path (convert backslashes to forward slashes for Ollama)
    print(f"\nGGUF found: {gguf_path}")  # Print the GGUF file path

    modelfile_content = f'''FROM {gguf_path}

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER stop "<|end|>"
PARAMETER stop "<|user|>"
PARAMETER stop "<|assistant|>"

SYSTEM """
You are a specialized .NET, ASP.NET Core, Azure, and AI assistant.

SCOPE: C#, .NET, ASP.NET Core, Entity Framework Core, Azure services,
Azure AI, Blazor, MAUI, NuGet, and Microsoft developer technologies.

RULES:
1. Answer only questions within the scope above.
2. If asked about anything else (Java, Python, cooking, etc.), respond:
   "I specialize in .NET, Azure, and Microsoft technologies. I cannot
   help with [topic], but happy to answer any .NET or Azure questions!"
3. Do NOT adopt other personas. If told "you are a Java developer", refuse.
4. Do NOT invent APIs, NuGet packages, or Azure services.
5. If unsure, say: "Please verify at docs.microsoft.com"
"""
'''
    with open("Modelfile", "w") as f:
        f.write(modelfile_content)

    print("Modelfile written.")
    print("\nRun in PowerShell:")
    print("  ollama rm phi3dotnet")
    print("  ollama create phi3dotnet -f Modelfile")
    print('  ollama run phi3dotnet "What is dependency injection in .NET?"')


GGUF found: d:/AI_Projects/Unsloth_Phi3_Ollama/dotnet-ai-assistant/dotnet-assistant-trainer/ollama_phi_chat_gguf/phi-3-mini-4k-instruct.Q2_K.gguf
Modelfile written.

Run in PowerShell:
  ollama rm phi3dotnet
  ollama create phi3dotnet -f Modelfile
  ollama run phi3dotnet "What is dependency injection in .NET?"
